
# Projet : Entrepôt de Données Hospitalier

## 1. Contexte

Dans le cadre du module *Systèmes d’Information Décisionnels* en Master 1 Informatique, ce projet vise à concevoir un entrepôt de données dédié à l’analyse des consultations médicales d’un hôpital.

Les établissements hospitaliers génèrent quotidiennement un volume important de données issues des consultations, des patients, des médecins et des services médicaux. Toutefois, ces données sont généralement stockées dans des bases opérationnelles qui ne sont pas optimisées pour l’analyse stratégique.

L’objectif est donc de transformer ces données brutes en un système décisionnel structuré permettant une analyse multidimensionnelle et une aide efficace à la prise de décision.

---

## 2. Objectifs

Ce projet poursuit plusieurs objectifs pédagogiques et techniques :

- Concevoir un entrepôt de données basé sur un modèle en étoile
- Identifier et structurer une table de faits et des tables de dimensions
- Mettre en place des hiérarchies au sein des dimensions
- Garantir la qualité et la cohérence des données avant leur intégration
- Réaliser des analyses OLAP (Online Analytical Processing)
- Concevoir des tableaux de bord interactifs à l’aide de l’outil BI Tableau

D’un point de vue métier, l’objectif est de permettre :
- L’analyse des coûts des consultations
- L’évaluation de l’activité des services hospitaliers
- L’étude de la fréquentation des médecins
- L’analyse temporelle des performances hospitalières

---

## 3. Description des données

Le jeu de données initial est composé de plusieurs fichiers CSV représentant :

### Table de faits : Consultations
- ID_Consultation (clé primaire)
- ID_Patient (clé étrangère)
- ID_Docteur (clé étrangère)
- ID_Service (clé étrangère)
- ID_Temps (clé étrangère)
- Coût
- Durée
- Diagnostic
- Traitement

### Tables de dimensions :
- Patients (informations démographiques)
- Docteurs (spécialité, ancienneté)
- Services (type de service, responsable)
- Temps (date, année, mois, trimestre, jour de la semaine)

### Granularité

La granularité du modèle est définie au niveau d’une consultation médicale.  
Chaque ligne de la table de faits représente un acte médical réalisé pour un patient donné, par un médecin donné, dans un service donné, à une date précise.

---

## 4. Méthodologie de prétraitement

Avant la construction de l’entrepôt de données, une phase de prétraitement est nécessaire afin d’assurer la qualité, la cohérence et l’intégrité des données.

Cette phase s’inscrit dans le processus ETL (Extract – Transform – Load).

Les étapes prévues sont :

1. Vérification des clés primaires (unicité et absence de valeurs nulles)
2. Contrôle de l’intégrité référentielle entre la table de faits et les dimensions
3. Détection et suppression des doublons
4. Traitement des valeurs manquantes
5. Détection des valeurs aberrantes (ex : coût ou durée négatifs)
6. Standardisation des formats (dates, variables numériques, catégories)

Cette étape est essentielle pour garantir la fiabilité des analyses décisionnelles et éviter toute incohérence dans les résultats produits ultérieurement.

# ***Importation des bibliothèques***

In [ ]:
import pandas as pd
import numpy as np


# ***Chargement des fichiers CSV***

In [ ]:
C = pd.read_csv("/content/consultations.csv")
D = pd.read_csv("/content/doctors.csv")
P = pd.read_csv("/content/patients.csv")
S = pd.read_csv("/content/services.csv")
T = pd.read_csv("/content/time.csv")

# ***Analyse exploratoire des données***


- L’analyse exploratoire des données constitue une étape essentielle avant la conception de l’entrepôt de données. Elle permet de comprendre la structure des tables, d’identifier les types de données, de détecter d’éventuelles valeurs manquantes, des doublons ou des incohérences.

- Cette étape garantit la qualité, la cohérence et l’intégrité des données avant leur intégration dans le modèle multidimensionnel (modèle en étoile ou en flocon).

### L’EDA comprend :

- L’examen des premières lignes des tables  
- L’analyse des types de données  
- Les statistiques descriptives(moyenne, minimum, maximum, écart-type).
- La détection des valeurs manquantes  
- La recherche de doublons  
- L’identification d’éventuelles valeurs aberrantes  
- La vérification de l’intégrité des clés primaires et étrangères  

Cette étape est indispensable pour assurer la fiabilité des analyses OLAP et des indicateurs décisionnels construits ultérieurement.

# Consultations

In [ ]:
C.head()


,ID_Consultation,ID_Patient,ID_Docteur,ID_Service,ID_Temps,Coût,Durée,Diagnostic,Traitement
0,1,277,49,8,259,198.09,100,Diabète,Médicaments contre la douleur
1,2,56,39,5,69,428.25,12,Diabète,Plâtre et suivi
2,3,252,62,1,150,143.03,32,Hypertension,Antihistaminiques
3,4,84,26,9,17,491.38,88,Asthme,Prescription d'antibiotiques
4,5,988,28,2,178,56.27,26,Allergie saisonnière,Inhalateurs prescrits


In [ ]:
C.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 9 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   ID_Consultation  10000 non-null  int64  
 1   ID_Patient       10000 non-null  int64  
 2   ID_Docteur       10000 non-null  int64  
 3   ID_Service       10000 non-null  int64  
 4   ID_Temps         10000 non-null  int64  
 5   Coût             10000 non-null  float64
 6   Durée            10000 non-null  int64  
 7   Diagnostic       10000 non-null  object 
 8   Traitement       10000 non-null  object 
dtypes: float64(1), int64(6), object(2)
memory usage: 703.3+ KB


In [ ]:
C.describe()

,ID_Consultation,ID_Patient,ID_Docteur,ID_Service,ID_Temps,Coût,Durée
count,10000.00000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000
mean,5000.50000,494.629300,50.477000,5.576200,183.275200,271.298334,65.391800
std,2886.89568,287.357684,28.721486,2.877155,105.496102,129.757824,32.126022
min,1.00000,1.000000,1.000000,1.000000,1.000000,50.010000,10.000000
25%,2500.75000,245.000000,26.000000,3.000000,92.750000,158.497500,37.000000
50%,5000.50000,492.500000,50.000000,6.000000,185.000000,269.785000,66.000000
75%,7500.25000,743.000000,76.000000,8.000000,274.000000,382.557500,93.000000
max,10000.00000,1000.000000,100.000000,10.000000,365.000000,499.960000,120.000000


In [ ]:
C.isnull().sum()

,0
ID_Consultation,0
ID_Patient,0
ID_Docteur,0
ID_Service,0
ID_Temps,0
Coût,0
Durée,0
Diagnostic,0
Traitement,0


In [ ]:
C.duplicated().sum()

np.int64(0)

In [ ]:
C["Coût"].min()

50.01

In [ ]:
C["Coût"].max()

499.96

In [ ]:
# Vérification de l’intégrité référentielle entre la table de faits `C` et les tables de dimensions (`P`, `D`, `S`, `T`) en comptant le nombre d'IDs valides pour chaque dimension :
print("Patients valides :", C["ID_Patient"].isin(P["ID_Patient"]).sum())
print("Docteurs valides :", C["ID_Docteur"].isin(D["ID_Docteur"]).sum())
print("Services valides :", C["ID_Service"].isin(S["ID_Service"]).sum())
print("Temps valides :", C["ID_Temps"].isin(T["ID_Temps"]).sum())   # La méthode isin() permet de vérifier l’existence des clés étrangères dans les tables de dimensions.

print("Total lignes table C :", len(C))
# Toutes les lignes sont cohérentes, donc aucune suppression n’est nécessaire avant la construction du modèle en étoile.

Patients valides : 10000
Docteurs valides : 10000
Services valides : 10000
Temps valides : 10000
Total lignes table C : 10000


# Docteurs

In [ ]:
D.head()

,ID_Docteur,Nom,Prénom,Spécialité,Ancienneté
0,1,Savage,Elizabeth,Généraliste,18
1,2,Camacho,Eddie,Cardiologie,15
2,3,Kennedy,Amanda,Cardiologie,39
3,4,Wilson,Keith,Radiologie,15
4,5,Holder,Kristy,Cardiologie,11


In [ ]:
D.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   ID_Docteur  100 non-null    int64 
 1   Nom         100 non-null    object
 2   Prénom      100 non-null    object
 3   Spécialité  100 non-null    object
 4   Ancienneté  100 non-null    int64 
dtypes: int64(2), object(3)
memory usage: 4.0+ KB


In [ ]:
D.describe()


,ID_Docteur,Ancienneté
count,100.000000,100.000000
mean,50.500000,21.040000
std,29.011492,11.949321
min,1.000000,1.000000
25%,25.750000,11.000000
50%,50.500000,21.500000
75%,75.250000,31.000000
max,100.000000,40.000000


In [ ]:
D.isnull().sum()


,0
ID_Docteur,0
Nom,0
Prénom,0
Spécialité,0
Ancienneté,0


In [ ]:
D.duplicated().sum()

np.int64(0)

In [ ]:
# Ancienneté : vérification qu’elle est ≥ 0 et réaliste (< 60 ans)
print("Ancienneté aberrante :", D[(D['Ancienneté'] < 0) | (D['Ancienneté'] > 60)].shape[0])

Ancienneté aberrante : 0


In [ ]:
print("Spécialités uniques :", D['Spécialité'].unique())

Spécialités uniques : ['Généraliste' 'Cardiologie' 'Radiologie' 'Pédiatrie' 'Dermatologie']


In [ ]:
print(D['Spécialité'].value_counts())

Spécialité
Cardiologie     23
Radiologie      21
Dermatologie    20
Généraliste     19
Pédiatrie       17
Name: count, dtype: int64


In [ ]:
print("Ancienneté uniques :", D['Ancienneté'].unique())

Ancienneté uniques : [18 15 39 11  6  3 33  1 28 23 21 26 32 31 17 35 14 37 40 38 20  7  2  8
 13  5 29 19 30 12 10 27 24 22  9]


# Patients

In [ ]:

P.head()


,ID_Patient,Nom,Prénom,Genre,Date_Naissance,Adresse,Code_Postal,Ville
0,1,Dixon,Joseph,Homme,1963-03-31,959 Brittany Circle,8333,Port Rebeccafort
1,2,Mclaughlin,Robert,Femme,1935-11-04,7450 Hannah Rest Apt. 565,46687,Aaronburgh
2,3,Doyle,Mallory,Homme,2008-07-13,2995 Kelli Viaduct Suite 576,52369,Meganhaven
3,4,Miller,Kyle,Femme,1929-10-04,0742 Krueger Roads Suite 055,45515,Hudsonchester
4,5,Thomas,Sandra,Femme,1974-02-18,5522 Lawrence Square,91295,Brownbury


In [ ]:
P.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   ID_Patient      1000 non-null   int64 
 1   Nom             1000 non-null   object
 2   Prénom          1000 non-null   object
 3   Genre           1000 non-null   object
 4   Date_Naissance  1000 non-null   object
 5   Adresse         1000 non-null   object
 6   Code_Postal     1000 non-null   int64 
 7   Ville           1000 non-null   object
dtypes: int64(2), object(6)
memory usage: 62.6+ KB


In [ ]:
P.describe()

,ID_Patient,Code_Postal
count,1000.000000,1000.000000
mean,500.500000,49464.049000
std,288.819436,28624.081758
min,1.000000,557.000000
25%,250.750000,25675.750000
50%,500.500000,49906.000000
75%,750.250000,72990.500000
max,1000.000000,99905.000000


In [ ]:
P.isnull().sum()

,0
ID_Patient,0
Nom,0
Prénom,0
Genre,0
Date_Naissance,0
Adresse,0
Code_Postal,0
Ville,0


In [ ]:
P.duplicated().sum()

np.int64(0)

# Services

In [ ]:

S.head()

,ID_Service,Nom_Service,Type_Service,Responsable
0,1,Service C1,Urgence,Dr. Smith
1,2,Service D2,Urgence,Dr. Le
2,3,Service B3,Consultation,Dr. Simmons
3,4,Service E4,Consultation,Dr. Olsen
4,5,Service A5,Chirurgie,Dr. Taylor


In [ ]:
S.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   ID_Service    10 non-null     int64 
 1   Nom_Service   10 non-null     object
 2   Type_Service  10 non-null     object
 3   Responsable   10 non-null     object
dtypes: int64(1), object(3)
memory usage: 452.0+ bytes


In [ ]:
S.describe()

,ID_Service
count,10.00000
mean,5.50000
std,3.02765
min,1.00000
25%,3.25000
50%,5.50000
75%,7.75000
max,10.00000


In [ ]:
S.isnull().sum()

,0
ID_Service,0
Nom_Service,0
Type_Service,0
Responsable,0


In [ ]:
S.duplicated().sum()

np.int64(0)

# Temps

In [ ]:

T.head()

,ID_Temps,Date,Année,Mois,Trimestre,Jour_Semaine
0,1,2024-01-01,2024,1,1,Monday
1,2,2024-01-02,2024,1,1,Tuesday
2,3,2024-01-03,2024,1,1,Wednesday
3,4,2024-01-04,2024,1,1,Thursday
4,5,2024-01-05,2024,1,1,Friday


In [ ]:
T.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 365 entries, 0 to 364
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   ID_Temps      365 non-null    int64 
 1   Date          365 non-null    object
 2   Année         365 non-null    int64 
 3   Mois          365 non-null    int64 
 4   Trimestre     365 non-null    int64 
 5   Jour_Semaine  365 non-null    object
dtypes: int64(4), object(2)
memory usage: 17.2+ KB


In [ ]:
T.describe()

,ID_Temps,Année,Mois,Trimestre
count,365.000000,365.0,365.000000,365.00000
mean,183.000000,2024.0,6.498630,2.50137
std,105.510663,0.0,3.448702,1.11834
min,1.000000,2024.0,1.000000,1.00000
25%,92.000000,2024.0,4.000000,2.00000
50%,183.000000,2024.0,7.000000,3.00000
75%,274.000000,2024.0,9.000000,3.00000
max,365.000000,2024.0,12.000000,4.00000


In [ ]:
T.isnull().sum()

,0
ID_Temps,0
Date,0
Année,0
Mois,0
Trimestre,0
Jour_Semaine,0


In [ ]:
T.duplicated().sum()

np.int64(0)

In [ ]:
# années, mois ou trimestres incohérents
print("Année aberrante :", T[(T['Année'] < 2000) | (T['Année'] > 2050)].shape[0])
print("Mois aberrant :", T[(T['Mois'] < 1) | (T['Mois'] > 12)].shape[0])

Année aberrante : 0
Mois aberrant : 0


# ***Prétraitement des dondonnées***

***Consultation***

In [ ]:
print(C)

      ID_Consultation  ID_Patient  ID_Docteur  ID_Service  ID_Temps    Coût  \
0                   1         277          49           8       259  198.09   
1                   2          56          39           5        69  428.25   
2                   3         252          62           1       150  143.03   
3                   4          84          26           9        17  491.38   
4                   5         988          28           2       178   56.27   
...               ...         ...         ...         ...       ...     ...   
9995             9996         585           5           6       197  223.98   
9996             9997         607          62           9       340  247.13   
9997             9998         751          29           4        51  115.18   
9998             9999         366          24           5       118  248.11   
9999            10000         760           4           7       191  337.95   

      Durée            Diagnostic                  

**Étape 1 : Création de DIM_DIAGNOSTIC**

In [ ]:
# On extrait les diagnostics uniques

dim_diagnostic = C[['Diagnostic']].drop_duplicates().reset_index(drop=True)

# Maintenant on crée un identifiant

dim_diagnostic['ID_Diagnostic'] = dim_diagnostic.index + 1

# Et réorganiser les colonnes :

dim_diagnostic = dim_diagnostic[['ID_Diagnostic','Diagnostic']]
dim_diagnostic

,ID_Diagnostic,Diagnostic
0,1,Diabète
1,2,Hypertension
2,3,Asthme
3,4,Allergie saisonnière
4,5,Migraine
5,6,Infection respiratoire
6,7,Fracture osseuse


**Pourquoi la colonne Diagnostic est devenue des chiffres ?**

C’est normal et c’est même le but. Au debut dans consultations :
Diagnostic
- Diabète
- Hypertension
- Asthme

Mais dans un entrepôt de données, la table de faits ne garde pas de texte. Elle garde des clés (IDs).

Donc :

Diagnostic	      devient                 ID_Diagnostic

Diabète	             →	                    1

Hypertension         →                     	2 ...

Et les textes restent dans DIM_DIAGNOSTIC. Donc le chiffre représente le diagnostic.

**Étape 2 : Reliement de DIM_DIAGNOSTIC à consultations**

In [ ]:
# Maintenant on ajoute ID_Diagnostic dans la table de faits.

C = C.merge(dim_diagnostic, on='Diagnostic', how='left')

# Puis on supprime la colonne texte :

C = C.drop(columns=['Diagnostic'])

C.head()

,ID_Consultation,ID_Patient,ID_Docteur,ID_Service,ID_Temps,Coût,Durée,Traitement,ID_Diagnostic
0,1,277,49,8,259,198.09,100,Médicaments contre la douleur,1
1,2,56,39,5,69,428.25,12,Plâtre et suivi,1
2,3,252,62,1,150,143.03,32,Antihistaminiques,2
3,4,84,26,9,17,491.38,88,Prescription d'antibiotiques,3
4,5,988,28,2,178,56.27,26,Inhalateurs prescrits,4


**Étape 3 : Création DIM_TRAITEMENT**

In [ ]:
# On extrait les traitements uniques depuis la table consultations

dim_traitement = C[['Traitement']].drop_duplicates().reset_index(drop=True)

# On Crée l’ID

dim_traitement['ID_Traitement'] = dim_traitement.index + 1

# Et on réorganise

dim_traitement = dim_traitement[['ID_Traitement','Traitement']]
dim_traitement

,ID_Traitement,Traitement
0,1,Médicaments contre la douleur
1,2,Plâtre et suivi
2,3,Antihistaminiques
3,4,Prescription d'antibiotiques
4,5,Inhalateurs prescrits
5,6,Régime et insuline
6,7,Prescription d'antihypertenseurs


**Étape 4 : Reliement DIM_TRAITEMENT à consultations**

In [ ]:
# On ajoute ID_Traitement dans consultations

C = C.merge(dim_traitement, on='Traitement', how='left')

# Supprimer la colonne texte Traitement

C = C.drop(columns=['Traitement'])
C.head()


,ID_Consultation,ID_Patient,ID_Docteur,ID_Service,ID_Temps,Coût,Durée,ID_Diagnostic,ID_Traitement
0,1,277,49,8,259,198.09,100,1,1
1,2,56,39,5,69,428.25,12,1,2
2,3,252,62,1,150,143.03,32,2,3
3,4,84,26,9,17,491.38,88,3,4
4,5,988,28,2,178,56.27,26,4,5


***Patients***

In [ ]:
# affichage de la table Patients
print(P)

     ID_Patient         Nom   Prénom  Genre Date_Naissance  \
0             1       Dixon   Joseph  Homme     1963-03-31   
1             2  Mclaughlin   Robert  Femme     1935-11-04   
2             3       Doyle  Mallory  Homme     2008-07-13   
3             4      Miller     Kyle  Femme     1929-10-04   
4             5      Thomas   Sandra  Femme     1974-02-18   
..          ...         ...      ...    ...            ...   
995         996       Gomez  Jessica  Femme     1942-03-05   
996         997      Farmer     Mary  Homme     1925-02-23   
997         998        Shaw    Susan  Homme     1930-02-18   
998         999    Mcintosh    Bruce  Femme     1964-01-16   
999        1000      Herman  Jillian  Femme     1978-01-29   

                           Adresse  Code_Postal             Ville  
0              959 Brittany Circle         8333  Port Rebeccafort  
1        7450 Hannah Rest Apt. 565        46687        Aaronburgh  
2     2995 Kelli Viaduct Suite 576        52369    

Étape 1 : Convertion de Date_Naissance en datetime

In [ ]:
import pandas as pd
from datetime import datetime

P['Date_Naissance'] = pd.to_datetime(P['Date_Naissance'])


**Pourquoi on a mis '2024-01-01'**

- Dans beaucoup de projets de données hospitalières ou OLAP, on fixe une date “snapshot”, par exemple le début de l’année, pour que tous les calculs soient cohérents.

- Si on utilise la date exacte du jour (datetime.today()), l’âge change tous les jours. Donc tes tableaux ou dashboards ne seront plus reproductibles exactement d’une session à l’autre.

- Ici '2024-01-01' sert juste à avoir un âge stable pour l’analyse.

Étape 2 :  Calcule de l’âge de chaque patient

In [ ]:
date_ref = pd.to_datetime('2024-01-01')  # date de référence
P['Age'] = (date_ref - P['Date_Naissance']).dt.days // 365

Étape 3 : Création les tranches d’âge

In [ ]:
def tranche_age(age):
    if age <= 17:
        return '0-17'
    elif age <= 35:
        return '18-35'
    elif age <= 50:
        return '36-50'
    elif age <= 65:
        return '51-65'
    else:
        return '66+'

P['Tranche_Age'] = P['Age'].apply(tranche_age)

In [ ]:
# Vérification du résultat
P[['ID_Patient','Date_Naissance','Age','Tranche_Age']].head(10)


,ID_Patient,Date_Naissance,Age,Tranche_Age
0,1,1963-03-31,60,51-65
1,2,1935-11-04,88,66+
2,3,2008-07-13,15,0-17
3,4,1929-10-04,94,66+
4,5,1974-02-18,49,36-50
5,6,1951-05-08,72,66+
6,7,1948-08-28,75,66+
7,8,1956-01-30,67,66+
8,9,1966-11-07,57,51-65
9,10,1953-05-30,70,66+


On utilisera Age et Tranche_Age dans Tableau pour :

- Analyser la distribution des patients par âge ou tranche d’âge

- Faire des filtres ou des hiérarchies (par exemple : Tranche_Age → Age)

- Croiser avec diagnostics, traitements ou services

In [ ]:
P.head()

,ID_Patient,Nom,Prénom,Genre,Date_Naissance,Adresse,Code_Postal,Ville,Age,Tranche_Age
0,1,Dixon,Joseph,Homme,1963-03-31,959 Brittany Circle,8333,Port Rebeccafort,60,51-65
1,2,Mclaughlin,Robert,Femme,1935-11-04,7450 Hannah Rest Apt. 565,46687,Aaronburgh,88,66+
2,3,Doyle,Mallory,Homme,2008-07-13,2995 Kelli Viaduct Suite 576,52369,Meganhaven,15,0-17
3,4,Miller,Kyle,Femme,1929-10-04,0742 Krueger Roads Suite 055,45515,Hudsonchester,94,66+
4,5,Thomas,Sandra,Femme,1974-02-18,5522 Lawrence Square,91295,Brownbury,49,36-50


***Docteur***

In [ ]:
D.head()

Pourquoi créer Catégorie_Ancienneté ?

On crée la variable Catégorie_Ancienneté à partir de Ancienneté afin de regrouper les médecins par niveau d’expérience. Cela permet de simplifier les analyses et les visualisations dans l’outil BI, par exemple comparer le nombre de consultations ou les coûts entre médecins juniors, confirmés, seniors ou experts.

Pourquoi on n'a pas supprimer Ancienneté car dans une dimension, on garde généralement :

   la valeur numérique détaillée

   la catégorie dérivée


In [ ]:
def niveau_experience(x):
    if x <= 5:
        return "Junior"
    elif x <= 15:
        return "Confirmé"
    elif x <= 30:
        return "Senior"
    else:
        return "Expert"

D["Niveau_Experience"] = D["Ancienneté"].apply(niveau_experience)

D.head()

***Temps***

Étape 1 : Vérification de la table temps

In [ ]:
print(T)

     ID_Temps        Date  Année  Mois  Trimestre Jour_Semaine
0           1  2024-01-01   2024     1          1       Monday
1           2  2024-01-02   2024     1          1      Tuesday
2           3  2024-01-03   2024     1          1    Wednesday
3           4  2024-01-04   2024     1          1     Thursday
4           5  2024-01-05   2024     1          1       Friday
..        ...         ...    ...   ...        ...          ...
360       361  2024-12-26   2024    12          4     Thursday
361       362  2024-12-27   2024    12          4       Friday
362       363  2024-12-28   2024    12          4     Saturday
363       364  2024-12-29   2024    12          4       Sunday
364       365  2024-12-30   2024    12          4       Monday

[365 rows x 6 columns]


Étape 2 : Transformation la colonne Date en format date

Nous allons ajouter 3 nouvelles colonnes :
- Semestre
- Saison
- Type_Jour

Puis vérifier que la hiérarchie est correcte.
La hiérarchie que nous allons utiliser dans Tableau est :

Année -> Semestre ->Trimestre
 -> Mois -> Date

Cette hiérarchie permettra dans Tableau :

- drill-down année → mois → jour

- analyse saisonnière

- comparaison semestre

- analyse semaine vs week-end

In [ ]:
# C’est important pour les manipulations temporelles

T['Date'] = pd.to_datetime(T['Date'])

# On ajoute les colonnes comme definit dans la modélisation:
# 1. Colonne Semestre
# Logique : mois 1 à 6 → semestre 1 et mois 7 à 12 → semestre 2
T['Semestre'] = T['Mois'].apply(lambda x: 1 if x <= 6 else 2)

# 2. Colonne Saison
# Règle classique :
# Mois	        Saison
# 12,1,2	      Hiver
# 3,4,5       	Printemps
# 6,7,8	        Été
# 9,10,11	      Automne
def saison(mois):
    if mois in [12,1,2]:
        return "Hiver"
    elif mois in [3,4,5]:
        return "Printemps"
    elif mois in [6,7,8]:
        return "Été"
    else:
        return "Automne"

T['Saison'] = T['Mois'].apply(saison)

# 3. Colonne Type_Jour
# On distingue : Semaine, Week-end

T['Type_Jour'] = T['Jour_Semaine'].apply(
    lambda x: "Weekend" if x in ["Saturday","Sunday"] else "Semaine"
)

# Vérification le résultat
T.head()

,ID_Temps,Date,Année,Mois,Trimestre,Jour_Semaine,Semestre,Saison,Type_Jour
0,1,2024-01-01,2024,1,1,Monday,1,Hiver,Semaine
1,2,2024-01-02,2024,1,1,Tuesday,1,Hiver,Semaine
2,3,2024-01-03,2024,1,1,Wednesday,1,Hiver,Semaine
3,4,2024-01-04,2024,1,1,Thursday,1,Hiver,Semaine
4,5,2024-01-05,2024,1,1,Friday,1,Hiver,Semaine


# **Exportation des fichiers propres**

In [ ]:
# Vérification des table
print("FACT_CONSULTATIONS")
print(C.head())
print("\nDIM_PATIENTS")
print(P.head())
print("\nDIM_DOCTEURS")
print(D.head())
print("\nDIM_SERVICES")
print(S.head())
print("\nDIM_TEMPS")
print(T.head())
print("\nDIM_DIAGNOSTIC")
print(dim_diagnostic.head())
print("\nDIM_TRAITEMENT")
print(dim_traitement.head())

FACT_CONSULTATIONS
   ID_Consultation  ID_Patient  ID_Docteur  ID_Service  ID_Temps    Coût  \
0                1         277          49           8       259  198.09   
1                2          56          39           5        69  428.25   
2                3         252          62           1       150  143.03   
3                4          84          26           9        17  491.38   
4                5         988          28           2       178   56.27   

   Durée  ID_Diagnostic  ID_Traitement  
0    100              1              1  
1     12              1              2  
2     32              2              3  
3     88              3              4  
4     26              4              5  

DIM_PATIENTS
   ID_Patient         Nom   Prénom  Genre Date_Naissance  \
0           1       Dixon   Joseph  Homme     1963-03-31   
1           2  Mclaughlin   Robert  Femme     1935-11-04   
2           3       Doyle  Mallory  Homme     2008-07-13   
3           4      Mille

In [ ]:
# Les fichiers CSV propres
# Chemin pour le dossier
import os
chemin = "C:/Users/marte/OneDrive/Desktop/COURS UNIVERSITÉ LUMIÈRE 2/ProjetBI/DonneesFinal/"
# dossier s'il n'existe pas
if not os.path.exists(chemin):
    os.makedirs(chemin)

C.to_csv("fact_consultations.csv", index=False)
P.to_csv("dim_patients.csv", index=False)
D.to_csv("dim_doctors.csv", index=False)
S.to_csv("dim_services.csv", index=False)
T.to_csv("dim_temps.csv", index=False)
dim_diagnostic.to_csv("dim_diagnostic.csv", index=False)
dim_traitement.to_csv("dim_traitement.csv", index=False)

print("\n Tous les fichiers CSV ont été exportés et sont prêts pour Tableau.")


 Tous les fichiers CSV ont été exportés et sont prêts pour Tableau.


Ces fichiers contiennent uniquement les colonnes nécessaires, les IDs pour les relations, et toutes les données prétraitées.